# Milestone 2: Data Ingestion

This notebook helps you ingest course materials for the Socratic AI tutor.

## Data Sources

You should document all URLs you ingest here for the project requirements.

In [8]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "src"))

from socratic.ingestion import scrape_course_website, process_pdf, process_slides
from socratic.models import Resource
import json

## 1. Course Website Scraping

Scrape the main course website and related pages.

In [9]:
# Scrape main course website
base_urls = [
    "https://pantelis.github.io/courses/ai/in-person.html",
    # Add more URLs as needed
]

all_resources = []

for url in base_urls:
    print(f"Scraping {url}...")
    resources = scrape_course_website(
        base_url=url,
        max_depth=2  # Adjust as needed
    )
    all_resources.extend(resources)
    print(f"  Collected {len(resources)} pages\n")

print(f"\nTotal web resources: {len(all_resources)}")

Scraping https://pantelis.github.io/courses/ai/in-person.html...
  Collected 126 pages


Total web resources: 126


## 2. PDF Processing

Process PDF files from the course.

In [10]:
# Process PDFs
pdf_dir = project_root / "data" / "raw" / "pdfs"
pdf_dir.mkdir(parents=True, exist_ok=True)

print(f"Looking for PDFs in: {pdf_dir}")
print("Add your PDF files to this directory.\n")

pdf_files = list(pdf_dir.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files:")

pdf_resources = []
for pdf_path in pdf_files:
    try:
        print(f"  Processing: {pdf_path.name}")
        resource = process_pdf(pdf_path)
        pdf_resources.append(resource)
        all_resources.append(resource)
        print(f"    ✅ {len(resource.content)} chars extracted")
    except Exception as e:
        print(f"    ❌ Error: {e}")

print(f"\nSuccessfully processed {len(pdf_resources)} PDFs")

Looking for PDFs in: ./data/raw/pdfs
Add your PDF files to this directory.

Found 0 PDF files:

Successfully processed 0 PDFs


## 3. PowerPoint Slides Processing

Process PPTX files from the course.

In [11]:
# Process slides
slides_dir = project_root / "data" / "raw" / "slides"
slides_dir.mkdir(parents=True, exist_ok=True)

print(f"Looking for PPTX files in: {slides_dir}")
print("Add your PowerPoint files to this directory.\n")

pptx_files = list(slides_dir.glob("*.pptx"))
print(f"Found {len(pptx_files)} PPTX files:")

slides_resources = []
for pptx_path in pptx_files:
    try:
        print(f"  Processing: {pptx_path.name}")
        resource = process_slides(pptx_path)
        slides_resources.append(resource)
        all_resources.append(resource)
        print(f"    ✅ {resource.metadata['num_slides']} slides extracted")
    except Exception as e:
        print(f"    ❌ Error: {e}")

print(f"\nSuccessfully processed {len(slides_resources)} slide decks")

Looking for PPTX files in: ./data/raw/slides
Add your PowerPoint files to this directory.

Found 0 PPTX files:

Successfully processed 0 slide decks


## 4. Save Ingested Resources

Save all resources for later use.

In [12]:
# Save resources
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

resources_file = output_dir / "ingested_resources.json"

# Convert to dict for JSON serialization
resources_data = [r.model_dump() for r in all_resources]

with open(resources_file, 'w', encoding='utf-8') as f:
    json.dump(resources_data, f, indent=2)

print(f"✅ Saved {len(all_resources)} resources to {resources_file}")

✅ Saved 126 resources to ./data/processed/ingested_resources.json


## 5. Document Ingested URLs

Create a manifest of all ingested URLs (required for M2 deliverable).

In [13]:
# Create URL manifest
manifest = {
    "web_resources": [],
    "pdf_resources": [],
    "slide_resources": [],
    "total_count": len(all_resources)
}

for resource in all_resources:
    entry = {
        "id": resource.id,
        "title": resource.title,
        "url": resource.url,
        "type": resource.type.value
    }
    
    if resource.type.value == "web":
        manifest["web_resources"].append(entry)
    elif resource.type.value == "pdf":
        manifest["pdf_resources"].append(entry)
    elif resource.type.value == "slide":
        manifest["slide_resources"].append(entry)

manifest_file = output_dir / "ingestion_manifest.json"
with open(manifest_file, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print(f"📄 Manifest saved to {manifest_file}")
print(f"\nSummary:")
print(f"  - Web pages: {len(manifest['web_resources'])}")
print(f"  - PDFs: {len(manifest['pdf_resources'])}")
print(f"  - Slides: {len(manifest['slide_resources'])}")
print(f"  - Total: {manifest['total_count']}")

📄 Manifest saved to ./data/processed/ingestion_manifest.json

Summary:
  - Web pages: 126
  - PDFs: 0
  - Slides: 0
  - Total: 126


## Display Sample Content

In [14]:
# Show sample resources
print("Sample Resources:\n")
for i, resource in enumerate(all_resources[:5]):
    print(f"{i+1}. [{resource.type.value.upper()}] {resource.title}")
    print(f"   URL: {resource.url}")
    print(f"   Content: {len(resource.content)} chars")
    if resource.content:
        preview = resource.content[:200].replace('\n', ' ')
        print(f"   Preview: {preview}...")
    print()

Sample Resources:

1. [WEB] Introduction to Artificial Intelligence – Engineering AI Agents
   URL: https://pantelis.github.io/courses/ai/in-person.html
   Content: 1940 chars
   Preview: Introduction to Artificial Intelligence Instructed by Pantelis Monogioudis, Ph.D and staff. What this course is all about Artificial Intelligence (AI) addresses one of the ultimate puzzles humans are ...

2. [WEB] Back2classroom – Engineering AI Agents
   URL: https://pantelis.github.io/index.html
   Content: 14072 chars
   Preview: Engineering AI Agents BOOK Foundations Training Deep Networks Perception Kinematics State Estimation Large Language Models Multimodal Reasoning Task Planning Global Planning Local Planning Markov Deci...

3. [WEB] Foundations – Engineering AI Agents
   URL: https://pantelis.github.io/book/foundations/index.html
   Content: 1089 chars
   Preview: The Learning Problem Statistical Learning Theory The supervised learning problem statement. Optimization Algorithms Optimization 

## Next Steps

Now that you've ingested the data, proceed to:
1. Notebook `02_build_knowledge_graph.ipynb` to build the knowledge graph (M3)
2. Or continue with `00_setup_and_test.ipynb` for the full pipeline